<details>
<summary>Phase 0 — Project setup in your repo</summary>

Perfect. This is the right flagship project to build first.

You’ve already designed it well. Now let’s turn it into something that looks like a real BA/BI implementation at Wyld, not just “a dashboard.” The winning pattern is:

**Source extracts → staging → conformance → marts → semantic model → report pages → reconciliation controls → publish/deploy**

That’s the job.

---

# Project 1 Build Plan

## Goal

Build a Power BI **Ops Command Center** that integrates:

* **Sales**
* **Operations / Inventory**
* **People / Labor**
* (plus a small **Finance reconciliation extract** for controls)

…and proves:

* cross-functional integration
* KPI governance
* QA/reconciliation discipline
* executive storytelling

---

# Phase 0 — Project setup in your repo

## Folder structure

Use this inside `project_01_wyld_ops_command_center/`:

```txt
project_01_wyld_ops_command_center/
  README.md

  data/
    source_extracts/
      sales/
      ops/
      people/
      finance/
    standardized/
    modeled/
    exceptions/
    sample/                     # tiny examples only (commit)
    raw/                        # gitignored

  sql/
    staging/
      stg_sales_distributor.sql
      stg_inventory_erp.sql
      stg_labor_payroll.sql
      stg_finance_actuals.sql
    conformance/
      int_sales_conformed.sql
      int_inventory_conformed.sql
      int_labor_conformed.sql
    marts/
      fact_sales.sql
      fact_inventory.sql
      fact_labor.sql
      dim_date.sql
      dim_product.sql
      dim_location.sql
      dim_channel.sql
      dim_employee_group.sql
      mart_reconciliation_controls.sql
      mart_anomaly_flags.sql
    validation/
      reconciliation_checks.sql
      semantic_model_test_queries.sql

  powerbi/
    wyld_ops_command_center.pbix
    semantic_model/
      model_design.md
      relationships_map.md
      dax_measure_catalog.md
      semantic_model_validation.md
      naming_conventions.md
      refresh_assumptions.md
    report_pages/
      page_inventory.md
      tooltip_narratives.md

  docs/
    source_register.md
    executive_walkthrough.md
    reporting_calendar.md
    stakeholder_notes.md
    reconciliation_log.md

  reports/
    executive_decks/
      ops_command_center_outline.md
    ad_hoc_requests/
      ad_hoc_request_template.md
      example_margin_drop_analysis.md
    scheduled_exports/
      ops_kpi_weekly_export_sample.csv
```

---


<details>
<summary>Phase 1 — Simulate the “messy source extracts” (very important)</summary>

# Phase 1 — Simulate the “messy source extracts” (very important)

This is the realism layer. Don’t skip it.

## 1) Create the source intake scenario

Pretend you receive these on a schedule:

* `sales_distributor_extract.csv` (daily/weekly sales)
* `inventory_erp_snapshot.csv` (daily inventory + shipments)
* `labor_hours_payroll_export.xlsx` (weekly labor/headcount/overtime)
* `finance_actuals_summary.xlsx` (monthly finance totals for reconciliation)

If you want speed, use your synthetic enterprise database to generate/export these source-style files. That’s a smart move.

## 2) Intentionally include source mess

Add a few realistic issues:

* inconsistent column names (`Store ID` vs `store_id`)
* text dates in one file, actual dates in another
* trailing spaces in product names
* missing `location_id` in some rows
* one duplicate row in sales
* a finance monthly total that differs slightly from modeled sales

This gives you something to QA/reconcile, which is exactly what they care about.

## 3) Build `docs/source_register.md`

Document each source in a table with:

* Source name
* Domain
* File type
* Refresh cadence
* Grain
* Join keys
* Owner
* Known issues
* Notes

This is a quiet power move. It shows process maturity.

---


<details><summary>1) simulate the data:</summary>

### Files created

* [Download the generator pack](sandbox:/mnt/data/project1_fake_data_generator_pack.zip)
* [Script in repo](sandbox:/mnt/data/wyld-business-analyst/scripts/generate_project1_fake_data.py)
* [Method doc](sandbox:/mnt/data/wyld-business-analyst/project_01_wyld_ops_command_center/docs/fake_data_generation_method.md)

---

## How to actually create the fake data

From your repo root, run:

```bash
python scripts/generate_data.py
```

That script will generate:

### Modeled truth layer

`project_01_wyld_ops_command_center/data/modeled/`

* dims + facts (`fact_sales`, `fact_inventory`, `fact_labor`)

### Messy source extracts

`project_01_wyld_ops_command_center/data/source_extracts/`

* `sales_distributor_extract.csv`
* `inventory_erp_snapshot.csv`
* `labor_hours_payroll_export.xlsx`
* `finance_actuals_summary.xlsx`

### Small git-friendly sample

`1_ops_command_center/data/sample/`

---

## Why this script is solid (and portfolio-useful)

It does the grown-up version of fake data:

* builds **clean dimensions first**
* generates **cross-functional facts** that actually relate to each other
* creates **messy source extracts** from the truth layer
* injects **controlled defects** (duplicates, missing IDs, inconsistent text, date gaps, finance drift)

That means your QA/QC + reconciliation pages won’t be fake theater — they’ll catch real simulated problems.

---

## Recommended first run (lighter / faster)

If your machine groans, start smaller:

```bash
python scripts/generate_data.py --products 40 --locations 10
```

Then scale up later.

---

## Next move after generation

1. Point your SQL staging queries at `data/source_extracts/...`
2. Build conformance + marts
3. Use `data/modeled/*.csv` as your reconciliation benchmark
4. Build Power BI semantic model + reconciliation page

This is exactly how you make a fake dataset look like a real analyst workspace instead of a random number carnival.

---
---

Excellent question. This is where the project goes from “nice mockup” to “damn, this feels real.”

You want fake data that is:

* **plausible**
* **cross-functional** (Sales / Ops / People / Finance tie together)
* **messy enough** to prove QA/QC skills
* **small enough** to manage in Power BI
* **structured enough** for a star schema

The trick is to generate it like a tiny company, not random numbers in a blender.

---

# The best way to produce fake data

Use a **hybrid approach**:

1. **Create clean dimension tables first** (products, locations, channels, employee groups, dates)
2. **Generate a “gold” modeled dataset** (fact_sales, fact_inventory, fact_labor) with realistic relationships
3. **Export fake source extracts** from the gold dataset in messy formats (CSV/XLSX)
4. **Intentionally inject issues** into the source extracts (duplicates, missing IDs, naming inconsistencies)
5. **Generate a finance summary extract** that mostly matches but has slight deltas (for reconciliation)

That gives you both:

* a truth layer (what the model should be)
* a messy intake layer (what the analyst receives)

That’s exactly real life.

---

# What you should generate for Project 1

## Dimensions (clean, stable)

Generate these first:

### `dim_date`

* Daily rows for 2–3 years (enough for trends + seasonality)
* Include:

  * `date_key`
  * `full_date`
  * year, quarter, month, week
  * `month_start_date`
  * `is_weekend`

### `dim_product`

Make ~40–120 products (good sweet spot)
Include:

* `product_key`
* `source_product_code` (important for conformance joins)
* `product_name`
* `brand`
* `category` (Gummies, Beverages, etc.)
* `pack_size`
* `is_active`

### `dim_location`

Make ~10–40 locations/accounts
Include:

* `location_key`
* `source_location_code`
* `location_name`
* `state/province`
* `region`
* `location_type` (retail / distributor / facility)
* `is_active`

### `dim_channel`

Small list:

* Retail
* Wholesale
* E-commerce (if you want)
* Distributor

### `dim_employee_group`

~5–12 groups
Include:

* `employee_group_key`
* `department_name`
* `team_name`

Examples:

* Operations / Fulfillment
* Operations / Warehouse
* Sales / Field
* G&A / Corporate
* Finance / FP&A

---

## Facts (the “truth” modeled layer)

## 1) `fact_sales` (daily grain)

Use grain:
**date + product + location + channel**

Fields:

* `date_key`
* `product_key`
* `location_key`
* `channel_key`
* `units_sold`
* `gross_sales_amount`
* `discount_amount`
* `net_sales_amount`
* `cogs_amount`
* `order_count`
* `customer_count`

### Make it realistic

Use simple drivers:

* base demand by product
* location multiplier
* weekday/weekend effect
* seasonality (summer bump, holiday promo)
* channel mix
* occasional promotions (higher units, lower margin)

This creates believable:

* revenue trends
* margin shifts
* VWAP changes
* anomalies

---

## 2) `fact_inventory` (daily snapshot)

Grain:
**date + product + location**

Fields:

* `date_key`
* `product_key`
* `location_key`
* `on_hand_units`
* `received_units`
* `shipped_units`
* `requested_units` (for fill rate)
* `backordered_units`
* `in_stock_flag`
* `promised_ship_date` (optional if simulating delivery)
* `actual_ship_date` (optional)

### Make it realistic

Tie inventory to sales:

* `shipped_units` should correlate with sales demand
* `on_hand_units` declines from shipments and increases from receipts
* occasional stockouts
* occasional delayed receipts (to cause stockouts)
* backorders spike when on-hand too low

This is how you earn Fill Rate / Stockout / DIO credibility.

---

## 3) `fact_labor` (daily or weekly)

Grain:
**date + location + employee_group**

Fields:

* `date_key`
* `location_key`
* `employee_group_key`
* `labor_hours`
* `overtime_hours`
* `headcount`
* `terminations` (optional, monthly)
* `hires` (optional, monthly)
* `labor_cost_amount`

### Make it realistic

Tie labor to operations/sales:

* labor hours should loosely scale with shipments/sales volume
* overtime should spike during promo periods or stockout recovery
* headcount should change slowly
* attrition should be occasional, not constant

This gives you:

* Revenue per Labor Hour
* Overtime %
* Attrition %
* Hiring Velocity

---

## 4) `finance_actuals_summary` (fake source extract only)

This should be a **monthly summarized Excel-style file**, not a fact table.

Fields:

* `month_start`
* `metric_name`
* `amount`
* `currency_code`

Metrics:

* net_sales
* gross_sales
* cogs
* gross_margin
* labor_cost
* maybe inventory_value (optional)

### Important:

Make it **almost** match your modeled totals (not exact)

* Example: ±0.5% to ±1.5% drift in some months
* This powers your reconciliation page

That is very realistic and very impressive.

---

# How to generate it (practical method)

## Recommended workflow

Use **Python + pandas** to generate the fake data.

Why:

* easy to create realistic distributions
* easy to inject controlled issues
* easy to export CSV/XLSX
* easy to reuse across projects

You already have `scripts/generate_demo_data.py` in the scaffold — perfect place.

---

# Generation strategy (the robust way)

## Step 1) Generate dimensions

Create clean dims and write them to:

* `data/modeled/` (truth layer)
* optionally also to `data/sample/` (small subset for Git)

## Step 2) Generate “truth” facts

Create:

* `fact_sales.csv`
* `fact_inventory.csv`
* `fact_labor.csv`

Write to:

* `data/modeled/`

These are your modeled target tables.

## Step 3) Create source extracts from truth facts

Transform the truth tables into messy source-like files:

* `sales_distributor_extract.csv`
* `inventory_erp_snapshot.csv`
* `labor_hours_payroll_export.xlsx`
* `finance_actuals_summary.xlsx`

Write to:

* `data/source_extracts/sales/`
* `data/source_extracts/ops/`
* `data/source_extracts/people/`
* `data/source_extracts/finance/`

## Step 4) Inject source issues (controlled mess)

Examples:

* duplicate 0.5–1% of sales rows
* blank `location_code` in 0.2% rows
* product names with extra spaces / casing changes
* one invalid negative inventory row
* one week missing from labor extract
* finance summary drift in one month

This gives your QA and reconciliation something to catch.

---

# Recommended scale for Project 1 (don’t overdo it)

You want enough data to look legit but not melt Power BI.

## Good target size

For a strong portfolio build:

* **Date range:** 18–24 months
* **Products:** 50–80
* **Locations:** 12–20
* **Channels:** 2–4
* **Employee groups:** 6–10

This usually yields roughly:

* `fact_sales`: 100k–400k rows (great)
* `fact_inventory`: 100k–300k rows
* `fact_labor`: 10k–50k rows

That’s plenty.

If your laptop starts wheezing, reduce locations/products before reducing date range.

---

# Make the data *look* enterprise (tiny details that matter)

These details make fake data feel real:

## Sales realism

* Different products have different price bands
* Different categories have different margins
* Promo periods increase units but lower margin
* Weekends shift channel mix
* Some locations are bigger than others

## Ops realism

* Stockouts cluster on high-volume SKUs
* Receipts happen in batches (not perfectly daily)
* Fill rate worsens when stockouts rise
* Backorders are rare but meaningful

## People realism

* Labor scales with volume, but imperfectly
* Overtime spikes during demand peaks
* Headcount changes slowly
* Attrition/hiring mostly monthly events

## Finance realism

* Monthly summary is rounded
* One or two metrics drift slightly from modeled totals
* Finance labels don’t perfectly match BI labels (`Net Sales` vs `net_sales`)

This is chef-level realism.

---

# How to organize fake data files in your repo

## Commit vs ignore

### Commit ✅

* small samples in `data/sample/`
* schemas/data dictionaries
* generation script
* maybe one tiny “demo week” source extract

### Don’t commit ❌

* full large generated files in `data/source_extracts/` or `data/modeled/`
* local DBs

Use your `.gitignore` to keep it clean.

---

# Suggested fake source schemas for your project

These align nicely with your staging SQL names.

## `sales_distributor_extract.csv` (source-facing, a little messy)

Columns (example):

* `sale_date`
* `sku`
* `product_name`
* `store_id`
* `channel`
* `qty`
* `gross_sales`
* `net_sales`
* `cogs`
* `orders`
* `customers`

(Matches your `stg_sales_distributor.sql` aliases nicely.)

## `inventory_erp_snapshot.csv`

* `snapshot_date`
* `sku`
* `site_code`
* `on_hand`
* `receipts`
* `shipments`
* `requested_units`
* `backordered_units`

## `labor_hours_payroll_export.xlsx`

* `work_date`
* `site_code`
* `department`
* `team`
* `hours_worked`
* `ot_hours`
* `employee_count`
* `labor_cost`

## `finance_actuals_summary.xlsx`

* `month_start`
* `metric_name`
* `actual_amount`
* `currency_code`

---

# How to inject “good mess” (examples)

Keep this controlled. You want *plausible* defects, not chaos spaghetti.

## Sales extract issues

* Duplicate 0.5% rows
* 0.3% null `store_id`
* `channel` values inconsistent (`Wholesale`, ` wholesale`, `WHOLESALE`)
* some `product_name` values with trailing spaces

## Inventory extract issues

* 20 rows with negative `on_hand`
* missing one day for one location
* `site_code` format inconsistent (`OR01` vs `or-01`)

## Labor extract issues

* one missing week for a team
* `team` renamed midstream (`Fulfillment` vs `Fulfilment`)
* one row with zero hours but nonzero cost

## Finance extract issues

* one month net sales off by +0.9%
* `metric_name` formatting inconsistent (`Gross Margin`, `gross_margin`)

These are exactly the kinds of things your project can “find and explain.”

---

# Best practice for reproducibility

Your generation script should support:

* a **fixed random seed** (important)
* configurable row volume
* configurable date range
* ability to regenerate exactly the same data

Example config block (in `scripts/generate_demo_data.py`):

* `SEED = 42`
* `START_DATE = "2024-01-01"`
* `END_DATE = "2025-12-31"`
* `N_PRODUCTS = 60`
* `N_LOCATIONS = 15`

That makes your project reproducible and interview-ready.

---

# What to build first in the fake data generator

Build in this order:

1. `dim_date`
2. `dim_product`
3. `dim_location`
4. `dim_channel`
5. `dim_employee_group`
6. `fact_sales`
7. `fact_inventory` (using sales demand as driver)
8. `fact_labor` (using sales/shipments as driver)
9. `finance_actuals_summary` (monthly rollup from facts + slight drift)
10. source extract exports with injected mess

That order creates believable relationships.

---

# Power move for your portfolio

Create a doc:
`project_01_wyld_ops_command_center/docs/fake_data_generation_method.md`

Include:

* what’s synthetic
* what relationships are intentionally modeled
* what defects are intentionally injected
* why (to demonstrate QA/QC, reconciliation, conformance)

This makes you look transparent and methodical.

---

# The quickest path for you specifically

Since you already have a simulated enterprise database bundle, the fastest route is:

1. **Reuse/export subsets** from that simulated DB as the “truth” facts/dims
2. Generate the **source extracts** from those tables
3. Inject controlled source mess
4. Build your SQL pipeline + Power BI on top

That saves time while still looking robust.

---

# One more nice touch

Put a tiny sample in `data/sample/`:

* 2 weeks
* 5 products
* 3 locations

That lets recruiters or reviewers inspect your project without downloading giant files.

---

You’re doing this the right way: not “random fake data,” but **synthetic operational data with business logic and QA failure modes**. That’s exactly the kind of detail that makes a hiring manager trust your work.

</details>

<details>
<summary>2) Fake Data generation method</summary>

# Fake Data Generation Method — Project 1

Use `scripts/generate_data.py` to generate **realistic synthetic data** for your Ops Command Center.

## What it creates
### Modeled truth layer (`data/modeled/`)
- `dim_date.csv`
- `dim_product.csv`
- `dim_location.csv`
- `dim_channel.csv`
- `dim_employee_group.csv`
- `fact_sales.csv`
- `fact_inventory.csv`
- `fact_labor.csv`

### Messy source extracts (`data/source_extracts/`)
- `sales/sales_distributor_extract.csv`
- `ops/inventory_erp_snapshot.csv`
- `people/labor_hours_payroll_export.xlsx`
- `finance/finance_actuals_summary.xlsx`

### Git-friendly sample subset (`data/sample/`)
Small 2-week subset for repo inspection.

## Why this is good for your portfolio
The script intentionally injects realistic issues:
- duplicates
- missing IDs
- inconsistent text/casing
- date gaps
- slight finance reconciliation drift

That gives your staging + QA/QC + reconciliation logic something real to catch.

## Run it
From the repo root:

```bash
python scripts/generate_project1_fake_data.py
```

Custom run (same script, different volume/date range):

```bash
python scripts/generate_project1_fake_data.py --start 2025-01-01 --end 2025-12-31 --seed 42 --products 60 --locations 15
```

</details>

<details>
<summary>Phase 2 — Staging layer (SQL standardization)</summary>

# Phase 2 — Staging layer (SQL standardization)

This is where you prove you can tame messy extracts.

## Objective

Standardize each source into clean, typed, predictable tables.

## Build these staging queries

### `stg_sales_distributor.sql`

Should:

* cast dates properly
* standardize product/location/channel columns
* trim whitespace
* cast numeric amounts
* create flags:

  * `is_bad_amount`
  * `is_missing_key`
  * `is_duplicate_candidate`

### `stg_inventory_erp.sql`

Should:

* standardize snapshot date
* split `received_units` and `shipped_units`
* ensure on-hand is numeric and nonnegative
* create `in_stock_flag`
* flag invalid negative inventory rows

### `stg_labor_payroll.sql`

Should:

* standardize labor date/week ending date
* cast labor hours, overtime hours, headcount
* normalize employee group/team names
* flag missing department/team

### `stg_finance_actuals.sql`

Should:

* standardize period (month)
* cast finance totals (sales/margin/etc.)
* map finance account labels to KPI categories
* keep source totals intact for later reconciliation

## Deliverable mindset

Each staging query should be “boring and reliable.” Staging is not where you get cute.

---

</details>

<details>
<summary>Phase 3 — Conformance layer (shared dimensions)</summary>

# Phase 3 — Conformance layer (shared dimensions)

This is the bridge from raw source chaos to a semantic model.

## Objective

Make Sales, Inventory, and Labor align to the same dimensions.

## Build conformed dimensions

Use shared dims:

* `dim_date`
* `dim_product`
* `dim_location`
* `dim_channel`
* `dim_employee_group`

You already have `dim_date.sql`, which is a huge head start.

## Build conformance SQL

### `int_sales_conformed.sql`

Map sales source rows to:

* `date_key`
* `product_key`
* `location_key`
* `channel_key`

Handle unknowns explicitly:

* use `-1` / `Unknown` rows in dims if needed
* keep an `unmapped_flag` for QA

### `int_inventory_conformed.sql`

Map inventory snapshots to:

* `date_key`
* `product_key`
* `location_key`

Normalize grain:

* one row per `date + location + product`

### `int_labor_conformed.sql`

Map labor to:

* `date_key`
* `location_key`
* `employee_group_key`

If labor comes weekly, document and apply allocation logic (e.g., weekly to daily, or keep weekly grain and model carefully). For this project, daily is easiest if synthetic.

---


<details>
<summary>Phase 4 — Mart layer (facts + controls)</summary>

# Phase 4 — Mart layer (facts + controls)

This is where you build the model tables Power BI will consume.

## Fact tables

Build:

* `fact_sales`
* `fact_inventory`
* `fact_labor`

Optional but strong:

* `fact_reconciliation_controls`
* `fact_anomaly_flags`

## Mart design rules (important)

Document grain in `model_design.md`:

* `fact_sales`: one row per **date + product + location + channel** (or transaction grain if available)
* `fact_inventory`: one row per **date + product + location snapshot**
* `fact_labor`: one row per **date + location + employee_group**

## Reconciliation mart (must-have)

### `mart_reconciliation_controls.sql`

Create a table with rows like:

* `check_name`
* `check_date`
* `domain`
* `source_total`
* `modeled_total`
* `delta_amount`
* `delta_pct`
* `tolerance_pct`
* `status` (Pass/Warning/Fail)

Examples:

* Sales source net total vs `fact_sales` net total
* Inventory source on-hand total vs `fact_inventory`
* Labor source hours vs `fact_labor`
* Finance monthly sales vs modeled monthly net sales

This powers your “Reconciliation & Data Health” page.

## Anomaly mart (hard-mode feature)

### `mart_anomaly_flags.sql`

Create daily/weekly anomaly flags for:

* margin drops
* stockout spikes
* labor productivity drops

Use simple logic:

* rolling mean + rolling std dev
* z-score threshold (e.g., abs(z) > 2)
* or IQR if easier in SQL

Columns:

* `metric_name`
* `date_key`
* `location_key` (optional)
* `value`
* `baseline_avg`
* `z_score`
* `is_anomaly`

This is very impressive without being overengineered.

---


<details>
<summary>Phase 5 — Validation and QA checks (SQL)</summary>

# Phase 5 — Validation and QA checks (SQL)

This is how you prove trustworthiness.

## Build `sql/validation/reconciliation_checks.sql`

Include checks like:

* no null keys in facts
* no duplicate rows at fact grain
* negative amounts not allowed (except explicitly allowed)
* source-vs-modeled totals within tolerance
* no future dates
* no missing dates in `dim_date`

## Build `semantic_model_test_queries.sql`

Create a tiny suite of test slices:

* sample month sales total by state
* inventory stockout rate for one state
* labor hours by location
* margin % sanity check

You’ll use these to validate the Power BI model outputs.

---
</details>

<details>
<summary>Phase 6 — Power BI semantic model (the real centerpiece)</summary>

# Phase 6 — Power BI semantic model (the real centerpiece)

This is where a lot of portfolios get mushy. Yours should be explicit and governed.

## 1) Load only modeled tables

In Power BI, import:

* dims
* facts
* reconciliation mart
* anomaly mart

Do **not** build the final model from raw extracts inside Power BI. (You can still use Power Query for light shaping, but the point is to show SQL modeling discipline.)

## 2) Build the star schema

Relationships:

* `dim_date` → all facts on `date_key`
* `dim_product` → `fact_sales`, `fact_inventory`
* `dim_location` → all facts
* `dim_channel` → `fact_sales`
* `dim_employee_group` → `fact_labor`

Rules:

* one-to-many from dimensions to facts
* single-direction filters (where possible)
* avoid many-to-many
* avoid bi-directional unless truly necessary

Document this in:

* `powerbi/semantic_model/model_design.md`
* `relationships_map.md`

## 3) DAX measure governance (super important)

Organize measures into folders (display folders if using Tabular Editor; otherwise naming prefixes).

### Base measures

* `[Units Sold]`
* `[Gross Sales]`
* `[Net Sales]`
* `[Discount $]`
* `[COGS]`
* `[Labor Hours]`
* `[Overtime Hours]` (if modeled)
* `[Headcount]`

### Derived KPIs

* `[Gross Margin $]`
* `[Gross Margin %]`
* `[AOV]`
* `[Units per Transaction]`
* `[Net VWAP]`
* `[Revenue per Labor Hour]`
* `[Stockout Rate]`
* `[DIO]`
* `[Fill Rate]`
* `[On-Time Delivery %]` (if available/simulated)
* `[Cycle Time]` (if available/simulated)
* `[Attrition %]` (if you include people events)
* `[Hiring Velocity]` (if you include hiring synthetic table)

### Comparison measures

* `[Net Sales vs Prior Period $]`
* `[Net Sales vs Prior Period %]`
* `[Gross Margin % vs Prior Period bps]`
* `[Stockout Rate vs Prior Period bps]`

### QA / Controls measures

* `[Source Net Sales]`
* `[Modeled Net Sales]`
* `[Reconciliation Delta $]`
* `[Reconciliation Delta %]`
* `[Reconciliation Status]`
* `[Data Confidence Score]`

## 4) Build `dax_measure_catalog.md`

Document:

* measure name
* business definition
* DAX
* format
* page(s) used
* notes

This is one of the strongest artifacts in the whole repo.

---


<details>
<summary>Phase 7 — Report page architecture (exec-ready)</summary>

# Phase 7 — Report page architecture (exec-ready)

Build the report like a product, not a pile of visuals.

## Page 1 — Executive Overview

Audience: leadership / VP

Include:

* KPI cards (10-ish max)
* trend (net sales + margin + stockouts)
* top risks
* top opportunities
* slicers: date, region, channel

KPIs to show:

* Net Sales
* Gross Margin %
* AOV
* Stockout Rate
* Revenue per Labor Hour
* Attrition %
* Data Confidence Score

## Page 2 — Sales & Margin

Audience: Finance / Commercial

Include:

* sales trend
* gross margin trend
* product mix
* channel mix
* net VWAP trend
* waterfall (optional)
* anomaly flags for margin

## Page 3 — Ops & Inventory

Audience: Ops / Supply Chain

Include:

* fill rate
* stockout rate
* DIO
* backorder rate
* on-hand trend
* stockout heatmap by location/product
* anomaly flags for stockout spikes

## Page 4 — People & Productivity

Audience: People / Ops / Finance

Include:

* labor hours trend
* revenue per labor hour
* overtime %
* headcount trend
* attrition %
* hiring velocity
* location/team comparison

## Page 5 — Reconciliation & Data Health (must-have)

Audience: analyst / finance controls / leadership trust page

Include:

* source vs modeled totals by domain
* variance amount / %
* pass/warn/fail indicators
* freshness timestamp
* DQ warnings count
* Data Confidence Score card

This page is pure gold for your portfolio.

## Page 6 — Diagnostics / Drillthrough

Audience: analysts / power users

Include:

* detail table by product/location/date
* exception rows
* anomaly details
* drillthrough target from other pages

---


<details>
<summary>Phase 8 — Narrative tooltips (plain-English DAX)</summary>

# Phase 8 — Narrative tooltips (plain-English DAX)

This is the storytelling flex.

## Build narrative measures for tooltips/cards

Examples:

* “Net Sales increased X% vs prior period, driven by [top region/product], partially offset by [margin/stockout issue].”
* “Stockout rate worsened to X%, concentrated in [state] and [product family].”
* “Revenue per labor hour improved by X due to stronger sales volume and stable labor hours.”

Store logic in `powerbi/report_pages/tooltip_narratives.md`.

Even if the DAX isn’t poetry, the pattern shows you can communicate.

---


<details>
<summary>Phase 9 — Deployment (Power BI + portfolio)</summary>

# Phase 9 — Deployment (Power BI + portfolio)

You asked “build and deploy,” so here’s the practical deploy path.

## A) Local build (development)

* Use SQLite/Postgres locally for SQL modeling
* Export modeled tables to CSV (or connect directly if using Postgres)
* Build `wyld_ops_command_center.pbix`
* Validate against SQL test queries

## B) Power BI publish (portfolio deployment)

If you have Power BI Desktop + Service:

1. Save `.pbix` in `powerbi/`
2. Publish to Power BI Service (workspace)
3. Configure dataset refresh (manual is fine for portfolio)
4. Create a dashboard pin set (optional)
5. Export screenshots for repo README/docs

## C) Portfolio deployment artifacts (very important)

In the repo, include:

* `docs/executive_walkthrough.md` (6–8 slide script)
* screenshots in `docs/screenshots/`
* `powerbi/semantic_model/*` docs
* `sql/` pipeline files
* `docs/source_register.md`
* `docs/reporting_calendar.md`

This proves the project is not just a binary PBIX.

## D) Optional but excellent

Record a 2-minute demo:

* open with business context
* show Overview page
* show Reconciliation page
* show one drillthrough
* end with “what actions I’d recommend”

That’s chef’s kiss indeed.

---


<details>
<summary>build sequence</summary>

# Suggested build sequence (best order, low chaos)

Here’s the exact order I’d build it in:

1. **Source register + fake extracts**
2. **Staging SQL**
3. **Conformance SQL**
4. **Fact/dim marts**
5. **Validation + reconciliation SQL**
6. **Power BI model relationships**
7. **Base DAX measures**
8. **Derived KPI measures**
9. **Reconciliation page**
10. **Executive page**
11. **Ops / Sales / People pages**
12. **Narrative tooltips**
13. **Docs + walkthrough**
14. **Publish PBIX + screenshots**

This order keeps you from designing visuals before the data model is stable (the classic BI trap).

---

# KPI set recommendation (tight, high-impact)

Keep it to ~12 for the flagship page set:

## Sales / Finance

* Net Sales
* Gross Margin %
* AOV
* Units per Transaction
* Net VWAP

## Ops / Inventory

* Fill Rate
* Stockout Rate
* DIO
* Backorder Rate
* On-Time Delivery %

## People / Productivity

* Revenue per Labor Hour
* Overtime %
* Attrition %
* Hiring Velocity

## Trust / Controls

* Data Confidence Score

You don’t need all 15 on one page. Spread them by domain.

---

# Robust proof signals hiring managers notice

These are the things that make this project feel “real”:

* `source_register.md` (you understand source systems)
* staging/conformance SQL layers (you can model data, not just chart it)
* reconciliation page (you care about trust)
* DAX measure catalog (you govern metrics)
* semantic model validation doc (you test your model)
* executive walkthrough (you can communicate)

That combo is nasty good for this job.

---

# One final strategic note

For portfolio realism, it’s totally fine to use:

* public/synthetic sales + ops + people data
* your simulated Wyld-style schema
* fake finance extract for reconciliation

Just be explicit in docs:

* what’s synthetic
* what assumptions you made
* what the project is proving

That transparency makes you look stronger, not weaker.

---

You’re not building “a dashboard.”
You’re building a **business analysis operating system demo**. That’s exactly how you stand out for this role.

</details>